<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/Lecture_3/Lecture_3_Convolutional_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Глава 1. Пример 1: один вход, одно ядро (свёртка 1D)

Прежде чем строить глубокие свёрточные сети, полезно разобрать простейшую операцию, в которой виден каждый шаг вычислений. Эта глава — «эталонный пример» для свёрток: на нём удобно проверить, что вы понимаете прямой проход, цепное правило с разделением весов (weight sharing) и правило обновления. Дальше, в следующих главах, мы будем ссылаться на формулы из этой главы как на базовые. К каждой части главы прилагается код на Python, который порождает соответствующую схему.

## 1.1. Зачем нужен этот пример

Одномерная свёртка с одним ядром — это, по сути, композиция скользящего скалярного произведения и нелинейности:

$$
z_i = b + \sum_{m=1}^{K} w_m \, x_{i+m-1}, \qquad a_i = \sigma(z_i).
$$

В этом примере всего четыре обучаемых параметра: три веса ядра $w_1, w_2, w_3$ и одно смещение $b$. Их так мало, что все производные можно выписать в явном виде и проследить каждую стрелку на графе вычислений. Именно поэтому этот пример хорош как первая иллюстрация:

- **прямой проход** здесь — это три применения скалярного произведения;
- **обратный проход** — это суммирование градиентов по всем позициям, где побывало ядро;
- **обновление** — четыре вычитания (так как параметров четыре).

Разобравшись в этом, вы без труда перенесёте ту же логику на двумерные свёртки с сотнями каналов: изменится число стрелок и размерность, но структура вычислений останется той же.

## 1.2. Архитектура и обозначения

Сеть (точнее, один свёрточный слой) имеет **один входной вектор**, **одно ядро** и **один выходной вектор**. Обозначим:

| Символ | Смысл |
|---|---|
| $x$ | входной сигнал, вектор длины $n$ |
| $w_m$ | вес ядра (фильтра), $m=1,\dots,K$ |
| $b$ | смещение (bias), одно число |
| $K$ | длина ядра (размер окна) |
| $z_i$ | преактивация $i$-го выхода |
| $a_i$ | активация $i$-го выхода |
| $\sigma$ | функция активации (в примерах — сигмоида) |
| $t_i$ | истинный ответ |
| $L$ | функция потерь |
| $\eta$ | шаг обучения (learning rate) |

Итого **четыре обучаемых параметра**: $w_1$, $w_2$, $w_3$, $b$. На рис. 1.1 изображена соответствующая схема: входные синие блоки, скользящее желтое ядро, красная рамка текущего окна и серые блоки выходов.



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch


def draw_architecture():
    """Рис. 1.1: архитектура свёртки 1D (один вход, одно ядро)."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 7)
    ax.axis('off')

    # 1. Входной вектор (x)
    x_vals = ['x₁', 'x₂', 'x₃', 'x₄', 'x₅']
    for i, val in enumerate(x_vals):
        rect = patches.Rectangle((1 + i * 1.2, 2), 1, 1,
                                 facecolor='#AEDFF7', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.text(3.4, 3.6, r'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # 2. Ядро (w)
    for i, val in enumerate(['w₁', 'w₂', 'w₃']):
        rect = patches.Rectangle((1 + i * 1.2, 4.5), 1, 1,
                                 facecolor='#FFE29A', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 5.0, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.text(2.7, 6.0, r'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # Стрелки от ядра к первому окну
    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 3.05), xytext=(1.5 + i * 1.2, 4.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    # Рамка вокруг первого окна
    window_rect = patches.Rectangle((0.8, 1.7), 3.6, 1.6, fill=False,
                                    edgecolor='#E74C3C', linestyle='--',
                                    lw=2.5, zorder=5)
    ax.add_patch(window_rect)
    ax.text(2.6, 1.3, 'Окно 1', ha='center', fontsize=11,
            color='#E74C3C', fontweight='bold')

    # 3. Выход (z)
    for i, val in enumerate(['z₁', 'z₂', 'z₃']):
        rect = patches.Rectangle((9 + i * 1.5, 2), 1, 1,
                                 facecolor='#E8E8E8', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(9.5 + i * 1.5, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.text(10.5, 3.5, r'Выход $z$ (длина 3)',
            ha='center', fontsize=12, style='italic')

    # Стрелка от окна к z1 (с изгибом)
    arrow = FancyArrowPatch((4.4, 2.0), (8.9, 2.5),
                            arrowstyle='-|>', mutation_scale=20, lw=2.5,
                            color='#2C3E50',
                            connectionstyle='arc3,rad=-0.2', zorder=2)
    ax.add_patch(arrow)

    ax.text(6.6, 1.5, 'Скалярное\nпроизведение',
            ha='center', fontsize=11, color='#2C3E50')

    # Смещение (b)
    ax.annotate('', xy=(9.5, 1.95), xytext=(9.5, 0.5),
                arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5),
                zorder=2)
    ax.text(9.8, 1.0, r'Смещение $b$',
            fontsize=12, color='#1F6FB2', fontweight='bold')

    # 4. Активация
    rect = patches.Rectangle((13.5, 1.0), 1.2, 3.0,
                             facecolor='#B8E0B8', edgecolor='black',
                             lw=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(14.1, 2.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)

    # Стрелка от z3 к активации
    ax.annotate('', xy=(13.45, 2.5), xytext=(12.5, 2.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    ax.text(7.5, 0.0,
            r'Рис. 1.1. Свёртка 1D: окно из 3 элементов и ядро дают первый выход $z_1$',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


draw_architecture()


## 1.3. Прямой проход

Прямой проход (**forward pass**) — это вычисление выхода свёртки по заданному входу $x$. Оно выполняется строго последовательно, от входа к выходу; каждый следующий шаг опирается на результат предыдущего.

### Шаг 1. Линейная комбинация (преактивация)

Ядро «накладывается» на первые три элемента входа и вычисляет взвешенную сумму со смещением:

$$
z_1 = b + w_1 x_1 + w_2 x_2 + w_3 x_3.
$$

Здесь $z_1$ — это **преактивация**: значение до применения нелинейности. Затем окно сдвигается на одну позицию вправо (по умолчанию шаг $s=1$), и вычисляются остальные выходы. В общем виде для $i$-го выхода:

$$
z_i = b + \sum_{m=1}^{K} w_m \, x_{i+m-1}, \qquad i = 1, \dots, o,
$$

где длина выхода $o = n - K + 1$. Для нашего примера $n=5$, $K=3$, значит $o=3$:

$$
z_1 = b + w_1 x_1 + w_2 x_2 + w_3 x_3,
$$
$$
z_2 = b + w_1 x_2 + w_2 x_3 + w_3 x_4,
$$
$$
z_3 = b + w_1 x_3 + w_2 x_4 + w_3 x_5.
$$

Обратите внимание: веса $w_1, w_2, w_3$ используются во всех трех окнах. Это и есть **разделение весов** (weight sharing).

### Шаг 2. Активация

К каждой преактивации применяется нелинейная функция активации:

$$
a_i = \sigma(z_i).
$$

Для сигмоиды $\sigma(z) = \frac{1}{1 + e^{-z}}$. Величина $a_i$ — это **выход слоя**, который пойдёт дальше (например, на следующий свёрточный слой или в полносвязный слой).

### Схема прямого прохода

На рис. 1.2 те же шаги подписаны прямо над стрелками: слева направо вы видите, как окно вырезает часть входа, умножает на ядро, складывает со смещением и применяет активацию.




In [ ]:
def draw_forward_pass():
    """Рис. 1.2: архитектура + шаги прямого прохода."""
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis('off')

    # Вход
    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        rect = patches.Rectangle((1 + i * 1.2, 3), 1, 1,
                                 facecolor='#AEDFF7', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    # Ядро
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        rect = patches.Rectangle((1 + i * 1.2, 5.5), 1, 1,
                                 facecolor='#FFE29A', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 6.0, val, ha='center', va='center',
                fontsize=14, zorder=4)

    # Стрелки ядра
    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 4.05), xytext=(1.5 + i * 1.2, 5.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    # Рамка окна
    window_rect = patches.Rectangle((0.8, 2.7), 3.6, 1.6, fill=False,
                                    edgecolor='#E74C3C', linestyle='--',
                                    lw=2.5, zorder=5)
    ax.add_patch(window_rect)

    # Выход z
    for i, val in enumerate(['$z_1$', '$z_2$', '$z_3$']):
        rect = patches.Rectangle((9 + i * 1.5, 3), 1, 1,
                                 facecolor='#E8E8E8', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(9.5 + i * 1.5, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    # Стрелка к z1
    arrow = FancyArrowPatch((4.4, 3.0), (8.9, 3.5),
                            arrowstyle='-|>', mutation_scale=20, lw=2.5,
                            color='#2C3E50',
                            connectionstyle='arc3,rad=-0.2', zorder=2)
    ax.add_patch(arrow)

    # Активация
    rect = patches.Rectangle((13.5, 2.0), 1.2, 3.0,
                             facecolor='#B8E0B8', edgecolor='black',
                             lw=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(14.1, 3.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)
    ax.annotate('', xy=(13.45, 3.5), xytext=(12.5, 3.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    # Формулы
    ax.text(4.5, 1.8, '$z_1 = b + w_1x_1 + w_2x_2 + w_3x_3$',
            ha='center', fontsize=12, color='#2C3E50')
    ax.text(10.5, 1.8, r'$a_1 = \sigma(z_1)$',
            ha='center', fontsize=12, color='#2C3E50')

    ax.text(8.0, 7.5,
            'Рис. 1.2. Прямой проход: скалярное произведение и активация',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


draw_forward_pass()



## 1.4. Функция потерь

После прямого прохода у нас есть предсказания $a_1, a_2, a_3$. Сравним их с истинными ответами $t_1, t_2, t_3$ и введём функцию потерь — half mean squared error:

$$
L = \frac{1}{2} \sum_{i=1}^{o} (a_i - t_i)^2.
$$

Множитель $1/2$ введён исключительно для удобства: при дифференцировании он сокращается с двойкой из производной квадрата, и в результате получается простая формула $\partial L/\partial a_i = a_i - t_i$ (см. шаг 1 обратного прохода).

## 1.5. Обратный проход

Обратный проход (**backward pass**, или backpropagation) — это вычисление производных $L$ по всем параметрам. Мы применяем цепное правило, шаг за шагом, двигаясь **справа налево** по графу вычислений прямого прохода. Главное отличие от полносвязного слоя здесь в том, что каждый вес $w_m$ используется во всех окнах. Поэтому при обратном проходе градиенты **суммируются по всем позициям**.

### Шаг 1. Локальный градиент (дельта) для каждой позиции

Дифференцируем $L$ по выходу $a_i$, а затем по преактивации $z_i$:

$$
\frac{\partial L}{\partial a_i} = a_i - t_i, \qquad \frac{\partial a_i}{\partial z_i} = \sigma'(z_i).
$$

Вводим обозначение для «дельты» $i$-й позиции:

$$
\delta_i := \frac{\partial L}{\partial z_i} = \frac{\partial L}{\partial a_i} \cdot \frac{\partial a_i}{\partial z_i} = (a_i - t_i)\,\sigma'(z_i).
$$

Для сигмоиды $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. С этого момента $\delta_i$ — это удобное сокращение; оно полностью вычисляется из величин прямого прохода.

### Шаг 2. Градиент по смещению $b$

Смещение $b$ прибавляется к **каждой** преактивации $z_i$. По правилу суммы производных:

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{o} \frac{\partial L}{\partial z_i} \cdot \frac{\partial z_i}{\partial b}.
$$

Поскольку $z_i = b + \sum w_m x_{i+m-1}$, производная $\partial z_i/\partial b = 1$. Тогда:

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{o} \delta_i \cdot 1 = \delta_1 + \delta_2 + \delta_3.
$$

### Шаг 3. Градиенты по ядру $w$

Каждый вес $w_m$ участвует в каждом окне. Для примера выпишем **полную цепочку** для $w_1$:

$$
\frac{\partial L}{\partial w_1} = \sum_{i=1}^{o} \frac{\partial L}{\partial z_i} \cdot \frac{\partial z_i}{\partial w_1}.
$$

Из формулы $z_i$ видно, что $w_1$ всегда умножается на первый элемент $i$-го окна, то есть на $x_i$. Поэтому $\partial z_i/\partial w_1 = x_i$. Подставляем:

$$
\frac{\partial L}{\partial w_1} = \delta_1 x_1 + \delta_2 x_2 + \delta_3 x_3.
$$

Аналогично для $w_2$ (умножается на $x_{i+1}$) и $w_3$ (умножается на $x_{i+2}$):

$$
\frac{\partial L}{\partial w_2} = \delta_1 x_2 + \delta_2 x_3 + \delta_3 x_4,
$$
$$
\frac{\partial L}{\partial w_3} = \delta_1 x_3 + \delta_2 x_4 + \delta_3 x_5.
$$

В матричном виде это можно записать как $X^T \delta$, где $X$ — матрица из сдвинутых окон.

### Схема обратного прохода

Рис. 1.3 объединяет обе фазы: сверху сплошными синими стрелками идёт прямой проход, снизу пунктирными красными — обратный. От $\delta_1, \delta_2, \delta_3$ вниз отходят формулы для градиентов.




In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch


def draw_backward_pass():
    """Рис. 1.3: прямой (синие →) и обратный (красные ⇢) проход для свёртки."""
    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 9)
    ax.axis('off')

    ax.text(8.0, 8.5,
            'Рис. 1.3. Прямой (синие →) и обратный (красные ⇢) проход для свёртки',
            ha='center', fontsize=14, fontweight='bold')

    # --- 1. ЯДРО (w) — СВЕРХУ ---
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        rect = patches.Rectangle((2 + i * 1.5, 6.5), 1, 1,
                                 facecolor='#FFE29A', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(2.5 + i * 1.5, 7.0, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 7.8, 'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # --- 2. ВХОД (x) — В ЦЕНТРЕ ---
    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        rect = patches.Rectangle((1 + i * 1.3, 4), 1, 1,
                                 facecolor='#AEDFF7', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.3, 4.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 3.3, 'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # Прямой проход (синие сплошные): w → x
    for i in range(3):
        ax.annotate('', xy=(2.5 + i * 1.5, 5.05), xytext=(2.5 + i * 1.5, 6.45),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # Рамка первого окна (x1, x2, x3) — раньше забывали add_patch
    window_rect = patches.Rectangle((0.8, 3.7), 4.1, 1.6, fill=False,
                                    edgecolor='#2C3E50', linestyle='--',
                                    lw=2, zorder=5)
    ax.add_patch(window_rect)

    # --- 3. ДЕЛЬТЫ (δ) — СНИЗУ ---
    d_positions = [(2.5, 1.5), (4.5, 1.5), (6.5, 1.5)]
    for i, (cx, cy) in enumerate(d_positions):
        ax.add_patch(patches.Circle((cx, cy), 0.45,
                                    facecolor='#FADBD8', edgecolor='#E74C3C',
                                    lw=2, zorder=3))
        # ВАЖНО: {i+1} без удвоения — иначе подставится literal "{i+1}"
        ax.text(cx, cy, rf'$\delta_{i+1}$',
                ha='center', va='center', fontsize=14,
                color='#C0392B', zorder=4)

    ax.text(4.5, 0.6, r'Ошибки (дельты) $\delta$',
            ha='center', fontsize=12, style='italic', color='#C0392B')

    # Прямой проход (синие сплошные): x → δ
    for cx in (2.5, 4.5, 6.5):
        ax.annotate('', xy=(cx, 2.05), xytext=(cx, 3.95),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # --- 4. ОБРАТНЫЙ ПРОХОД (красные пунктирные): δ → w ---
    backward = [
        ((2.1, 1.5), (2.1, 6.4),  0.2),
        ((2.5, 1.9), (3.1, 6.4),  0.2),
        ((4.5, 1.9), (3.5, 6.4),  0.0),
        ((4.5, 1.9), (4.5, 6.4),  0.0),
        ((6.5, 1.9), (4.9, 6.4), -0.2),
        ((6.9, 1.5), (5.9, 6.4), -0.2),
    ]
    for (x0, y0), (x1, y1), rad in backward:
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1),
                                     arrowstyle='-|>', mutation_scale=15,
                                     lw=1.5, color='#E74C3C',
                                     linestyle='--',
                                     connectionstyle=f'arc3,rad={rad}'))

    # --- 5. ФОРМУЛЫ ГРАДИЕНТОВ (справа) ---
    formula_x = 10.0
    ax.text(formula_x, 6.8, 'Градиенты (результат обратного прохода):',
            fontsize=12, fontweight='bold')

    grads = [
        (5.8, r'$\frac{\partial L}{\partial w_1} = \delta_1 x_1 + \delta_2 x_2 + \delta_3 x_3$'),
        (4.5, r'$\frac{\partial L}{\partial w_2} = \delta_1 x_2 + \delta_2 x_3 + \delta_3 x_4$'),
        (3.2, r'$\frac{\partial L}{\partial w_3} = \delta_1 x_3 + \delta_2 x_4 + \delta_3 x_5$'),
        (1.8, r'$\frac{\partial L}{\partial b} = \delta_1 + \delta_2 + \delta_3$'),
    ]
    for y, s in grads:
        ax.text(formula_x, y, s, fontsize=12, color='#B9770E',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF9E7',
                          edgecolor='#E67E22', lw=1.5))

    plt.tight_layout()
    plt.show()


draw_backward_pass()



## 1.6. Обновление параметров

Заключительный шаг — собственно обучение. Все четыре параметра обновляются по правилу градиентного спуска с шагом обучения $\eta$. Ключевой момент, который часто упускают: **производные в этом правиле берутся в той самой точке пространства параметров, в которой сеть сейчас находится**, то есть после прямого прохода с текущими весами. Формально, если обозначить текущее состояние параметров через

$$
\theta^{(t)} = \bigl(w_1^{(t)},\, w_2^{(t)},\, w_3^{(t)},\, b^{(t)}\bigr),
$$

а индекс $t$ — номер шага обучения, то

$$
w_m^{(t+1)} := w_m^{(t)} - \eta \left.\frac{\partial L}{\partial w_m}\right|_{\theta^{(t)}}, \qquad b^{(t+1)} := b^{(t)} - \eta \left.\frac{\partial L}{\partial b}\right|_{\theta^{(t)}}.
$$

### Что означает «в точке $\theta^{(t)}$»

Когда мы пишем, например,

$$
\left.\frac{\partial L}{\partial w_1}\right|_{\theta^{(t)}} = \delta_1^{(t)} x_1 + \delta_2^{(t)} x_2 + \delta_3^{(t)} x_3,
$$

все величины в правой части — $\delta_i^{(t)}$ — **вычислены при текущих значениях параметров** $w_1^{(t)}, \dots, b^{(t)}$, то есть взяты из последнего прямого прохода. Поэтому градиентный спуск идёт итеративно:

1. **Прямой проход** с текущими $\theta^{(t)}$ даёт $z_i^{(t)}$, $a_i^{(t)}$.
2. **Обратный проход** с этими величинами даёт $\delta_i^{(t)}$, а затем и все четыре градиента — **в точке $\theta^{(t)}$**.
3. **Шаг обновления** сдвигает параметры в направлении, противоположном градиенту.
4. Цикл повторяется; теперь производные будут браться уже в точке $\theta^{(t+1)}$.

## 1.7. Один полный шаг обучения на числах

Чтобы формулы перестали быть абстрактными, проделаем один шаг алгоритма вручную. Возьмём:

- вход $x = (1, 0, 2, -1, 3)$;
- истинные ответы $t = (1, 0, 1)$;
- начальные параметры $w_1 = 0.5$, $w_2 = -0.5$, $w_3 = 0.25$, $b = 0.1$;
- шаг обучения $\eta = 0.1$;
- $\sigma(z) = 1/(1 + e^{-z})$, $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$.

### Прямой проход

$$
z_1 = 0.1 + 0.5(1) - 0.5(0) + 0.25(2) = 1.1,
$$
$$
z_2 = 0.1 + 0.5(0) - 0.5(2) + 0.25(-1) = -1.15,
$$
$$
z_3 = 0.1 + 0.5(2) - 0.5(-1) + 0.25(3) = 2.35.
$$

$$
a_1 = \sigma(1.1) \approx 0.75026,
$$
$$
a_2 = \sigma(-1.15) \approx 0.24049,
$$
$$
a_3 = \sigma(2.35) \approx 0.91293.
$$

Потеря:

$$
L = \tfrac{1}{2}\bigl[(0.75026 - 1)^2 + (0.24049 - 0)^2 + (0.91293 - 1)^2\bigr] \approx 0.06390.
$$

### Обратный проход

$$
\delta_1 = (0.75026 - 1) \cdot 0.75026(1 - 0.75026) \approx -0.04680,
$$
$$
\delta_2 = (0.24049 - 0) \cdot 0.24049(1 - 0.24049) \approx 0.04393,
$$
$$
\delta_3 = (0.91293 - 1) \cdot 0.91293(1 - 0.91293) \approx -0.00692.
$$

Градиенты:

$$
\frac{\partial L}{\partial b} = -0.04680 + 0.04393 - 0.00692 = -0.00979,
$$
$$
\frac{\partial L}{\partial w_1} = (-0.04680)(1) + (0.04393)(0) + (-0.00692)(2) = -0.06064,
$$
$$
\frac{\partial L}{\partial w_2} = (-0.04680)(0) + (0.04393)(2) + (-0.00692)(-1) = 0.09478,
$$
$$
\frac{\partial L}{\partial w_3} = (-0.04680)(2) + (0.04393)(-1) + (-0.00692)(3) = -0.15829.
$$

### Обновление

$$
w_1 \leftarrow 0.5 - 0.1(-0.06064) = 0.50606,
$$
$$
w_2 \leftarrow -0.5 - 0.1(0.09478) = -0.50948,
$$
$$
w_3 \leftarrow 0.25 - 0.1(-0.15829) = 0.26583,
$$
$$
b \leftarrow 0.1 - 0.1(-0.00979) = 0.10098.
$$

Если теперь выполнить прямой проход с новыми весами, потеря немного уменьшится.

## 1.8. Полная программа

Соберём всё вместе: класс `TinyConv1D` реализует прямой проход, обратный проход и шаг обучения.




In [ ]:
import math

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch


# ===========================================================================
# 1. Математика
# ===========================================================================
def sigmoid(z: float) -> float:
    """Численно устойчивая логистическая функция."""
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)            # z < 0 ⇒ ez ∈ (0, 1], переполнения нет
    return ez / (1.0 + ez)


def sigmoid_prime_from_a(a: float) -> float:
    """σ'(z) через уже посчитанное a = σ(z)."""
    return a * (1.0 - a)


# ===========================================================================
# 2. Модель
# ===========================================================================
class TinyConv1D:
    """Свёртка 1D: одно ядро, stride=1, padding=0, сигмоида, half-MSE."""

    def __init__(self, w, b: float):
        self.w = list(w)
        self.b = float(b)

    def __repr__(self) -> str:
        return f'TinyConv1D(w={self.w}, b={self.b:.5f})'

    # --- прямой проход -----------------------------------------------------
    def forward(self, x):
        n, K = len(x), len(self.w)
        if K > n:
            raise ValueError(f'Ядро длиннее входа: K={K} > n={n}')
        o = n - K + 1
        z = [0.0] * o
        a = [0.0] * o
        for i in range(o):
            z[i] = self.b + sum(self.w[m] * x[i + m] for m in range(K))
            a[i] = sigmoid(z[i])
        return z, a

    # --- потеря ------------------------------------------------------------
    @staticmethod
    def loss(a, t):
        """L = ½ Σ (a_i − t_i)²."""
        return 0.5 * sum((ai - ti) ** 2 for ai, ti in zip(a, t))

    # --- обратный проход ---------------------------------------------------
    def backward(self, x, t, z, a):
        """Возвращает (dw, db, delta). δ уже включает σ'(z)."""
        n, K = len(x), len(self.w)
        o = n - K + 1

        delta = [(a[i] - t[i]) * sigmoid_prime_from_a(a[i]) for i in range(o)]
        db = sum(delta)
        dw = [sum(delta[i] * x[i + m] for i in range(o)) for m in range(K)]
        return dw, db, delta

    # --- шаг обучения ------------------------------------------------------
    def update(self, dw, db, eta: float):
        for m in range(len(self.w)):
            self.w[m] -= eta * dw[m]
        self.b -= eta * db


# ===========================================================================
# 3. Рис. 1.1 — архитектура свёртки 1D
# ===========================================================================
def draw_architecture():
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 7)
    ax.axis('off')

    # Вход x1..x5
    for i, val in enumerate(['x₁', 'x₂', 'x₃', 'x₄', 'x₅']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 2), 1, 1,
                                       facecolor='#AEDFF7', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.4, 3.6, r'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # Ядро w1..w3
    for i, val in enumerate(['w₁', 'w₂', 'w₃']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 4.5), 1, 1,
                                       facecolor='#FFE29A', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 5.0, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(2.7, 6.0, r'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # Стрелки ядро → окно
    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 3.05), xytext=(1.5 + i * 1.2, 4.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    # Рамка первого окна
    ax.add_patch(patches.Rectangle((0.8, 1.7), 3.6, 1.6, fill=False,
                                   edgecolor='#E74C3C', linestyle='--',
                                   lw=2.5, zorder=5))
    ax.text(2.6, 1.3, 'Окно 1', ha='center', fontsize=11,
            color='#E74C3C', fontweight='bold')

    # Выход z1..z3
    for i, val in enumerate(['z₁', 'z₂', 'z₃']):
        ax.add_patch(patches.Rectangle((9 + i * 1.5, 2), 1, 1,
                                       facecolor='#E8E8E8', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(9.5 + i * 1.5, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(10.5, 3.5, r'Выход $z$ (длина 3)',
            ha='center', fontsize=12, style='italic')

    # Скалярное произведение: окно → z1
    ax.add_patch(FancyArrowPatch((4.4, 2.0), (8.9, 2.5),
                                 arrowstyle='-|>', mutation_scale=20,
                                 lw=2.5, color='#2C3E50',
                                 connectionstyle='arc3,rad=-0.2', zorder=2))
    ax.text(6.6, 1.5, 'Скалярное\nпроизведение',
            ha='center', fontsize=11, color='#2C3E50')

    # Смещение b
    ax.annotate('', xy=(9.5, 1.95), xytext=(9.5, 0.5),
                arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5),
                zorder=2)
    ax.text(9.8, 1.0, r'Смещение $b$',
            fontsize=12, color='#1F6FB2', fontweight='bold')

    # Активация
    ax.add_patch(patches.Rectangle((13.5, 1.0), 1.2, 3.0,
                                   facecolor='#B8E0B8', edgecolor='black',
                                   lw=1.5, zorder=3))
    ax.text(14.1, 2.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)
    ax.annotate('', xy=(13.45, 2.5), xytext=(12.5, 2.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    ax.text(7.5, 0.0,
            r'Рис. 1.1. Свёртка 1D: окно из 3 элементов и ядро дают первый выход $z_1$',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


# ===========================================================================
# 4. Рис. 1.2 — прямой проход
# ===========================================================================
def draw_forward_pass():
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis('off')

    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 3), 1, 1,
                                       facecolor='#AEDFF7', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 5.5), 1, 1,
                                       facecolor='#FFE29A', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 6.0, val, ha='center', va='center',
                fontsize=14, zorder=4)

    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 4.05), xytext=(1.5 + i * 1.2, 5.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    ax.add_patch(patches.Rectangle((0.8, 2.7), 3.6, 1.6, fill=False,
                                   edgecolor='#E74C3C', linestyle='--',
                                   lw=2.5, zorder=5))

    for i, val in enumerate(['$z_1$', '$z_2$', '$z_3$']):
        ax.add_patch(patches.Rectangle((9 + i * 1.5, 3), 1, 1,
                                       facecolor='#E8E8E8', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(9.5 + i * 1.5, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.add_patch(FancyArrowPatch((4.4, 3.0), (8.9, 3.5),
                                 arrowstyle='-|>', mutation_scale=20,
                                 lw=2.5, color='#2C3E50',
                                 connectionstyle='arc3,rad=-0.2', zorder=2))

    ax.add_patch(patches.Rectangle((13.5, 2.0), 1.2, 3.0,
                                   facecolor='#B8E0B8', edgecolor='black',
                                   lw=1.5, zorder=3))
    ax.text(14.1, 3.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)
    ax.annotate('', xy=(13.45, 3.5), xytext=(12.5, 3.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    ax.text(4.5, 1.8, r'$z_1 = b + w_1x_1 + w_2x_2 + w_3x_3$',
            ha='center', fontsize=12, color='#2C3E50')
    ax.text(10.5, 1.8, r'$a_1 = \sigma(z_1)$',
            ha='center', fontsize=12, color='#2C3E50')

    ax.text(8.0, 7.5,
            'Рис. 1.2. Прямой проход: скалярное произведение и активация',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


# ===========================================================================
# 5. Рис. 1.3 — прямой (синие →) и обратный (красные ⇢) проход
# ===========================================================================
def draw_backward_pass():
    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 9)
    ax.axis('off')

    ax.text(8.0, 8.5,
            'Рис. 1.3. Прямой (синие →) и обратный (красные ⇢) проход',
            ha='center', fontsize=14, fontweight='bold')

    # --- Ядро сверху ---
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        ax.add_patch(patches.Rectangle((2 + i * 1.5, 6.5), 1, 1,
                                       facecolor='#FFE29A', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(2.5 + i * 1.5, 7.0, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 7.8, r'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # --- Вход в центре ---
    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        ax.add_patch(patches.Rectangle((1 + i * 1.3, 4), 1, 1,
                                       facecolor='#AEDFF7', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.3, 4.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 3.3, r'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # Прямой проход: w → x (синие сплошные)
    for i in range(3):
        ax.annotate('', xy=(2.5 + i * 1.5, 5.05), xytext=(2.5 + i * 1.5, 6.45),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # Рамка первого окна
    ax.add_patch(patches.Rectangle((0.8, 3.7), 4.1, 1.6, fill=False,
                                   edgecolor='#2C3E50', linestyle='--',
                                   lw=2, zorder=5))

    # --- Дельты снизу ---
    for i, (cx, cy) in enumerate([(2.5, 1.5), (4.5, 1.5), (6.5, 1.5)]):
        ax.add_patch(patches.Circle((cx, cy), 0.45,
                                    facecolor='#FADBD8', edgecolor='#E74C3C',
                                    lw=2, zorder=3))
        ax.text(cx, cy, rf'$\delta_{i+1}$',         # ← ОДИНАРНЫЕ { }
                ha='center', va='center', fontsize=14,
                color='#C0392B', zorder=4)
    ax.text(4.5, 0.6, r'Ошибки (дельты) $\delta$',
            ha='center', fontsize=12, style='italic', color='#C0392B')

    # Прямой проход: x → δ (синие сплошные)
    for cx in (2.5, 4.5, 6.5):
        ax.annotate('', xy=(cx, 2.05), xytext=(cx, 3.95),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # Обратный проход: δ → w (красные пунктирные)
    for (x0, y0), (x1, y1), rad in [
        ((2.1, 1.5), (2.1, 6.4),  0.2),
        ((2.5, 1.9), (3.1, 6.4),  0.2),
        ((4.5, 1.9), (3.5, 6.4),  0.0),
        ((4.5, 1.9), (4.5, 6.4),  0.0),
        ((6.5, 1.9), (4.9, 6.4), -0.2),
        ((6.9, 1.5), (5.9, 6.4), -0.2),
    ]:
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1),
                                     arrowstyle='-|>', mutation_scale=15,
                                     lw=1.5, color='#E74C3C',
                                     linestyle='--',
                                     connectionstyle=f'arc3,rad={rad}'))

    # --- Формулы градиентов справа ---
    ax.text(10.0, 6.8, 'Градиенты (результат обратного прохода):',
            fontsize=12, fontweight='bold')

    for y, s in [
        (5.8, r'$\frac{\partial L}{\partial w_1} = \delta_1 x_1 + \delta_2 x_2 + \delta_3 x_3$'),
        (4.5, r'$\frac{\partial L}{\partial w_2} = \delta_1 x_2 + \delta_2 x_3 + \delta_3 x_4$'),
        (3.2, r'$\frac{\partial L}{\partial w_3} = \delta_1 x_3 + \delta_2 x_4 + \delta_3 x_5$'),
        (1.8, r'$\frac{\partial L}{\partial b} = \delta_1 + \delta_2 + \delta_3$'),
    ]:
        ax.text(10.0, y, s, fontsize=12, color='#B9770E',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF9E7',
                          edgecolor='#E67E22', lw=1.5))

    plt.tight_layout()
    plt.show()


# ===========================================================================
# 6. Демонстрация §1.7 — один полный шаг обучения
# ===========================================================================
def demo_one_step():
    net = TinyConv1D(w=[0.5, -0.5, 0.25], b=0.1)
    x = [1.0, 0.0, 2.0, -1.0, 3.0]
    t = [1.0, 0.0, 1.0]
    eta = 0.1

    print('=' * 62)
    print('§1.7. Один полный шаг обучения на числах')
    print('=' * 62)
    print(f'  x   = {x}')
    print(f'  t   = {t}')
    print(f'  w⁰  = {net.w},  b⁰ = {net.b},  η = {eta}')
    print()

    z, a = net.forward(x)
    L = net.loss(a, t)
    dw, db, delta = net.backward(x, t, z, a)

    print('Прямой проход:')
    for i in range(len(z)):
        print(f'  z{i+1} = {z[i]:8.5f},  a{i+1} = {a[i]:8.5f}')
    print(f'  L  = {L:.5f}')
    print()

    print('Обратный проход:')
    print(f'  δ      = {[round(d, 5) for d in delta]}')
    print(f'  ∂L/∂b  = {db:8.5f}')
    for m in range(len(dw)):
        print(f'  ∂L/∂w{m+1} = {dw[m]:8.5f}')
    print()

    net.update(dw, db, eta)
    print('После шага:')
    print(f'  w¹ = {[round(w, 5) for w in net.w]}')
    print(f'  b¹ = {net.b:.5f}')
    print()
    print('Сверка с текстом §1.7:')
    print('  z = [1.1, -1.15, 2.35]')
    print('  δ = [-0.04680, 0.04393, -0.00692]')
    print('  w = [0.50606, -0.50948, 0.26583], b = 0.10098')
    print('  L ≈ 0.06390')


# ===========================================================================
# 7. Точка входа
# ===========================================================================
if __name__ == '__main__':
    draw_architecture()
    draw_forward_pass()
    draw_backward_pass()
    demo_one_step()



Если запустить `demo_one_step()`, выводятся ровно те числа, что мы посчитали вручную в §1.7. Это хороший способ убедиться, что ни в одной формуле не потерян множитель: если что-то не сходится с ручным расчётом, ошибка почти наверняка в обратном проходе.

## 1.9. Что мы получили

В этой главе на минимальном примере мы разобрали все три фазы обучения свёрточного слоя:

1. **Прямой проход** — скользящее окно, скалярное произведение с ядром и активация.
2. **Обратный проход** — вычисление $\delta_i$ и **суммирование градиентов по всем позициям**, где побывало ядро (из-за разделения весов).
3. **Обновление** — вычитание усреднённого градиента с шагом $\eta$, причём производные берутся **в текущей точке** $\theta^{(t)}$.

В следующих главах мы добавим **Stride (шаг)** и **Padding (добавление нулей)**, а затем перейдём к двумерным свёрткам (картинкам). Число стрелок на схемах вырастет, но структура останется той же: на каждом шаге мы двигаемся по графу вычислений, перенося ошибку назад через производные и суммируя вклады от всех окон. Формулы этой главы — тот строительный блок, из которого складывается обучение более глубоких свёрточных сетей.



# Глава 2. Пример 2: Stride и Padding — управляем размером выхода

В Главе 1 мы разобрали базовую свёртку, где окно сдвигалось ровно на 1 шаг ($stride=1$) и не выходило за границы входа ($padding=0$). Теперь сделаем следующий шаг: **добавим шаг (Stride) и дополнение нулями (Padding)**. Это позволит нам управлять размером выхода и понять, как ошибка распределяется, когда окно «перепрыгивает» через элементы или заходит за края.

## 2.1. Зачем нужен этот пример

Базовая свёртка из Главы 1 теряла информацию по краям: $x_1$ участвовал только в $z_1$, а $x_5$ — только в $z_3$. Кроме того, размер выхода всегда был меньше размера входа.

В реальных сетях это неудобно. Нам нужны инструменты, чтобы:
- **Stride (шаг)** — прореживать сигнал (уменьшать размерность), заставляя ядро «прыгать» через элементы.
- **Padding (отступ)** — сохранять информацию на границах и контролировать размер выхода (например, делать его равным входу), добавляя нули по краям.

В этом примере мы разберем, как эти два параметра меняют математику прямого и обратного прохода.

## 2.2. Архитектура и обозначения (Рис. 2.1)

Сеть имеет **один входной вектор**, **одно ядро** и **один выходной вектор**. Обозначения:

| Символ | Смысл |
|---|---|
| $x$ | входной сигнал, вектор длины $n=5$ |
| $w_m$ | вес ядра, $m=1,\dots,K$ ($K=3$) |
| $b$ | смещение (bias) |
| $s$ | шаг (stride), $s=2$ |
| $p$ | отступ (padding), $p=1$ |
| $x'$ | дополненный нулями вход |
| $z_i$ | преактивация $i$-го выхода |
| $a_i$ | активация $i$-го выхода |
| $t_i$ | истинный ответ |
| $\delta_i$ | локальный градиент |

Итого **четыре обучаемых параметра**: $w_1, w_2, w_3, b$. Длина выхода при $p=1, s=2$ остается $o=3$.

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

def draw_architecture_stride_pad():
    """Рис. 2.1: Свёртка с Stride=2 и Padding=1."""
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 16); ax.set_ylim(0, 8)
    ax.axis('off')

    # Вход с padding
    x_vals = ['$0$', '$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$', '$0$']
    colors = ['#D3D3D3', '#AEDFF7', '#AEDFF7', '#AEDFF7', '#AEDFF7', '#AEDFF7', '#D3D3D3']
    for i, (val, color) in enumerate(zip(x_vals, colors)):
        rect = patches.Rectangle((1 + i*1.3, 3), 1, 1, facecolor=color, edgecolor='black', lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i*1.3, 3.5, val, ha='center', va='center', fontsize=14, zorder=4)
    ax.text(5.4, 2.3, 'Входной сигнал $x$ с Padding = 1 (серые нули)', ha='center', fontsize=12, style='italic')

    # Ядро
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        rect = patches.Rectangle((1 + i*1.3, 5.5), 1, 1, facecolor='#FFE29A', edgecolor='black', lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i*1.3, 6.0, val, ha='center', va='center', fontsize=14, zorder=4)
    ax.text(2.8, 7.0, 'Ядро $w$', ha='center', fontsize=12, style='italic')

    # Стрелки от ядра
    for i in range(3):
        ax.annotate('', xy=(1.5 + i*1.3, 4.05), xytext=(1.5 + i*1.3, 5.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2), zorder=2)

    # Окно 1 (padding 0, x1, x2)
    rect1 = patches.Rectangle((0.8, 2.8), 3.9, 1.4, fill=False, edgecolor='#E74C3C', linestyle='--', lw=2.5, zorder=5)
    ax.add_patch(rect1)
    ax.text(2.8, 2.3, 'Окно 1', ha='center', fontsize=11, color='#E74C3C', fontweight='bold')

    # Окно 2 (x2, x3, x4) - сдвиг на 2 (Stride = 2)
    rect2 = patches.Rectangle((3.4, 2.8), 3.9, 1.4, fill=False, edgecolor='#2980B9', linestyle=':', lw=2.5, zorder=5)
    ax.add_patch(rect2)
    ax.text(5.4, 2.3, 'Окно 2', ha='center', fontsize=11, color='#2980B9', fontweight='bold')

    # Выход (z)
    for i, val in enumerate(['$z_1$', '$z_2$', '$z_3$']):
        rect = patches.Rectangle((11 + i*1.5, 3), 1, 1, facecolor='#E8E8E8', edgecolor='black', lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(11.5 + i*1.5, 3.5, val, ha='center', va='center', fontsize=14, zorder=4)
    ax.text(12.5, 4.5, 'Выход $z$ (длина 3)', ha='center', fontsize=12, style='italic')

    # Стрелки к z
    ax.add_patch(FancyArrowPatch((4.7, 2.8), (10.9, 3.2), arrowstyle='-|>', mutation_scale=20, lw=2, color='#E74C3C', connectionstyle="arc3,rad=-0.1", zorder=2))
    ax.add_patch(FancyArrowPatch((7.3, 2.8), (10.9, 3.8), arrowstyle='-|>', mutation_scale=20, lw=2, color='#2980B9', connectionstyle="arc3,rad=0.1", zorder=2))

    # Смещение
    ax.annotate('', xy=(11.5, 2.95), xytext=(11.5, 1.5),
                arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5), zorder=2)
    ax.text(11.8, 2.0, 'Смещение $b$', fontsize=12, color='#1F6FB2', fontweight='bold')

    ax.text(8.0, 0.2, 'Рис. 2.1. Stride = 2 (синяя рамка перепрыгивает через один элемент) и Padding = 1',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_architecture_stride_pad()
```

## 2.3. Прямой проход

Прямой проход выполняется слева направо. Из-за шага $s=2$ окна не перекрываются так сильно, как в Главе 1.

### Шаг 1. Линейная комбинация (преактивация)

Формула размера выхода с учетом $p$ и $s$:
$$
o = \left\lfloor \frac{n + 2p - K}{s} \right\rfloor + 1 = \left\lfloor \frac{5 + 2 - 3}{2} \right\rfloor + 1 = 3.
$$
Пусть $x'$ — это дополненный нулями вход: $x' = (0, x_1, x_2, x_3, x_4, x_5, 0)$. Тогда:
$$
z_i = b + \sum_{m=1}^{K} w_m \, x'_{(i-1)s + m}, \qquad i=1,2,3.
$$
Расписываем для каждого окна:
$$
z_1 = b + w_1(0) + w_2(x_1) + w_3(x_2),
$$
$$
z_2 = b + w_1(x_2) + w_2(x_3) + w_3(x_4),
$$
$$
z_3 = b + w_1(x_4) + w_2(x_5) + w_3(0).
$$

### Шаг 2. Активация

К каждой преактивации применяется сигмоида:
$$
a_i = \sigma(z_i).
$$

### Шаг 3. Функция потерь

$$
L = \frac{1}{2} \sum_{i=1}^{o} (a_i - t_i)^2.
$$

## 2.4. Обратный проход

Это самый важный раздел. Мы применяем цепное правило и **выписываем полные цепочки** для каждого параметра.

### Шаг 1. Локальный градиент (дельта)

$$
\delta_i := \frac{\partial L}{\partial z_i} = \frac{\partial L}{\partial a_i} \cdot \frac{\partial a_i}{\partial z_i} = (a_i - t_i)\,\sigma'(z_i).
$$

### Шаг 2. Градиент по смещению $b$

Смещение $b$ прибавляется к **каждому** $z_i$. Поэтому изменение $b$ влияет на $L$ через три пути. По правилу суммы производных выписываем полную цепочку:
$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z_1}\frac{\partial z_1}{\partial b} + \frac{\partial L}{\partial z_2}\frac{\partial z_2}{\partial b} + \frac{\partial L}{\partial z_3}\frac{\partial z_3}{\partial b}.
$$
Так как $z_i = b + \dots$, производная $\frac{\partial z_i}{\partial b} = 1$. Подставляем дельты:
$$
\frac{\partial L}{\partial b} = \delta_1 \cdot 1 + \delta_2 \cdot 1 + \delta_3 \cdot 1 = \delta_1 + \delta_2 + \delta_3.
$$

### Шаг 3. Градиенты по ядру $w$

Каждый вес $w_m$ участвует во всех трех окнах. Выпишем полную цепочку для $w_1$:
$$
\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial z_1}\frac{\partial z_1}{\partial w_1} + \frac{\partial L}{\partial z_2}\frac{\partial z_2}{\partial w_1} + \frac{\partial L}{\partial z_3}\frac{\partial z_3}{\partial w_1}.
$$
Смотрим, на что умножается $w_1$ в каждом окне: в $z_1$ это $0$, в $z_2$ это $x_2$, в $z_3$ это $x_4$. Подставляем дельты:
$$
\frac{\partial L}{\partial w_1} = \delta_1(0) + \delta_2(x_2) + \delta_3(x_4).
$$
Аналогично для $w_2$ (умножается на $x_1, x_3, x_5$) и $w_3$ (умножается на $x_2, x_4, 0$):
$$
\frac{\partial L}{\partial w_2} = \delta_1(x_1) + \delta_2(x_3) + \delta_3(x_5),
$$
$$
\frac{\partial L}{\partial w_3} = \delta_1(x_2) + \delta_2(x_4) + \delta_3(0).
$$

### Шаг 4. Градиент по входу $x$ (и отбрасывание padding)

Сначала считаем градиент по расширенному входу $x'$. Для каждого элемента $x'_j$ суммируем вклады от всех окон, которые его накрыли. Но так как $x'_1$ и $x'_7$ — это искусственные нули (padding), мы **отбрасываем** их градиенты. Нам нужны только $\partial L / \partial x_j$ для исходных $x_1 \dots x_5$.

## 2.5. Обновление параметров

Правило обновления стандартное, производные берутся в текущей точке $\theta^{(t)}$:
$$
w_m^{(t+1)} := w_m^{(t)} - \eta \left.\frac{\partial L}{\partial w_m}\right|_{\theta^{(t)}}, \qquad b^{(t+1)} := b^{(t)} - \eta \left.\frac{\partial L}{\partial b}\right|_{\theta^{(t)}}.
$$

## 2.6. Один полный шаг обучения на числах

Проверим формулы вручную. Возьмём:
- Вход: $x = (1, 2, 3, 4, 5)$;
- Ядро: $w = (0.1, 0.2, 0.3)$;
- Смещение: $b = 0.0$;
- Параметры: $p = 1, s = 2$;
- Цель: $t = (1, 1, 1)$;
- Шаг: $\eta = 0.1$.

### Прямой проход

$x' = (0, 1, 2, 3, 4, 5, 0)$.
$$
z_1 = 0 + 0.1(0) + 0.2(1) + 0.3(2) = 0.8, \qquad a_1 = \sigma(0.8) \approx 0.68997,
$$
$$
z_2 = 0 + 0.1(2) + 0.2(3) + 0.3(4) = 2.0, \qquad a_2 = \sigma(2.0) \approx 0.88080,
$$
$$
z_3 = 0 + 0.1(4) + 0.2(5) + 0.3(0) = 1.4, \qquad a_3 = \sigma(1.4) \approx 0.80218.
$$
Потеря: $L \approx 0.5 \cdot [(0.68997-1)^2 + (0.88080-1)^2 + (0.80218-1)^2] \approx 0.07717$.

### Обратный проход

Дельты:
$$
\delta_1 = (0.68997 - 1) \cdot 0.68997(1-0.68997) \approx -0.06631,
$$
$$
\delta_2 = (0.88080 - 1) \cdot 0.88080(1-0.88080) \approx -0.01251,
$$
$$
\delta_3 = (0.80218 - 1) \cdot 0.80218(1-0.80218) \approx -0.03139.
$$

Градиенты:
$$
\frac{\partial L}{\partial b} = -0.06631 - 0.01251 - 0.03139 = -0.11021.
$$
$$
\frac{\partial L}{\partial w_1} = (-0.06631)(0) + (-0.01251)(2) + (-0.03139)(4) = -0.15058.
$$
$$
\frac{\partial L}{\partial w_2} = (-0.06631)(1) + (-0.01251)(3) + (-0.03139)(5) = -0.26079.
$$
$$
\frac{\partial L}{\partial w_3} = (-0.06631)(2) + (-0.01251)(4) + (-0.03139)(0) = -0.18266.
$$

### Обновление

$$
w_1 \leftarrow 0.1 - 0.1(-0.15058) = 0.11506,
$$
$$
w_2 \leftarrow 0.2 - 0.1(-0.26079) = 0.22608,
$$
$$
w_3 \leftarrow 0.3 - 0.1(-0.18266) = 0.31827,
$$
$$
b \leftarrow 0.0 - 0.1(-0.11021) = 0.01102.
$$

## 2.7. Полная программа

```python
import math
import numpy as np

def sigmoid(z: float) -> float:
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)

def sigmoid_prime_from_a(a: float) -> float:
    return a * (1.0 - a)

class TinyConv1D_Stride_Pad:
    """Свёртка 1D: одно ядро, произвольные stride и padding."""
    def __init__(self, w, b: float, stride: int = 1, padding: int = 0):
        self.w = list(w)
        self.b = float(b)
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        x_padded = np.pad(x, (self.padding, self.padding), mode='constant', constant_values=0)
        n_padded = len(x_padded)
        K = len(self.w)
        o = (n_padded - K) // self.stride + 1
        
        z = [0.0] * o
        a = [0.0] * o
        
        for i in range(o):
            start = i * self.stride
            z[i] = self.b + sum(self.w[m] * x_padded[start + m] for m in range(K))
            a[i] = sigmoid(z[i])
            
        return z, a, x_padded

    def backward(self, x, x_padded, t, z, a):
        K = len(self.w)
        o = len(z)
        
        delta = [(a[i] - t[i]) * sigmoid_prime_from_a(a[i]) for i in range(o)]
        
        db = sum(delta)
        
        dw = [0.0] * K
        for m in range(K):
            for i in range(o):
                start = i * self.stride
                dw[m] += delta[i] * x_padded[start + m]
                
        dx_padded = np.zeros(len(x_padded))
        for i in range(o):
            start = i * self.stride
            for m in range(K):
                dx_padded[start + m] += delta[i] * self.w[m]
                
        dx = dx_padded[self.padding : len(x) + self.padding]
        
        return dw, db, dx, delta

    def update(self, dw, db, eta: float):
        for m in range(len(self.w)):
            self.w[m] -= eta * dw[m]
        self.b -= eta * db

# Демонстрация
if __name__ == '__main__':
    net = TinyConv1D_Stride_Pad(w=[0.1, 0.2, 0.3], b=0.0, stride=2, padding=1)
    x = [1.0, 2.0, 3.0, 4.0, 5.0]
    t = [1.0, 1.0, 1.0]
    eta = 0.1

    z, a, x_padded = net.forward(x)
    dw, db, dx, delta = net.backward(x, x_padded, t, z, a)

    print(f"z = {[round(val, 5) for val in z]}")
    print(f"delta = {[round(val, 5) for val in delta]}")
    print(f"dw = {[round(val, 5) for val in dw]}, db = {round(db, 5)}")
    
    net.update(dw, db, eta)
    print(f"w' = {[round(val, 5) for val in net.w]}, b' = {round(net.b, 5)}")
```

## 2.8. Что мы получили

В этой главе мы увидели, как Stride и Padding меняют индексацию в прямом проходе, но **не меняют логику обратного**. Градиенты по-прежнему суммируются по всем окнам (как в Главе 1), а добавленные нули (padding) просто отбрасываются при вычислении градиента по входу.


# Глава 3. Пример 3: Свёртка 2D (картинки)

В предыдущих двух главах мы разбирали одномерную свёртку (1D), где сигнал был строчкой. В реальном мире мы чаще всего имеем дело с изображениями, где данные имеют два пространственных измерения: высоту (Height) и ширину (Width). Теперь сделаем следующий шаг: **перейдем к двумерной свёртке (2D)**.

Эта глава — ключевая для понимания работы компьютерного зрения (CNN). Вы увидите, что математика прямого и обратного прохода остаётся абсолютно такой же, как в Главе 1, но добавляется второе измерение, и мы начинаем использовать **матрицы** (двумерные массивы) вместо векторов.

## 3.1. Зачем нужен этот пример

В 1D-свёртке ядро скользило слева направо. В 2D-свёртке мы имеем **входную матрицу** (например, изображение \(3 \times 3\) пикселя) и **ядро** (матрицу, например, \(2 \times 2\)). Ядро скользит по входу и вправо, и вниз.

Главные отличия 2D от 1D:
- **Двойное суммирование:** каждый выход теперь зависит от суммы произведений по высоте и ширине.
- **Ядро — матрица:** веса \(w_{m,n}\) теперь имеют два индекса: \(m\) (строка) и \(n\) (столбец).
- **Выход — матрица:** результат свёртки (feature map) имеет размеры \(H_{out} \times W_{out}\).

В этой главе мы разберем один канал, одно ядро, stride=1 и padding=0. Это «эталонный пример» для 2D.

## 3.2. Архитектура и обозначения (Рис. 3.1)

Слой имеет **один входной канал** (матрицу), **одно ядро** и **один выходной канал**.

| Символ | Смысл |
|---|---|
| \(X\) | входная матрица, размер \(H \times W\) (высота × ширина) |
| \(W\) | ядро (фильтр), размер \(K_h \times K_w\) |
| \(b\) | смещение (bias), одно число |
| \(Z\) | матрица преактиваций, размер \(H_{out} \times W_{out}\) |
| \(A\) | матрица активаций (выход), размер \(H_{out} \times W_{out}\) |
| \(T\) | матрица истинных ответов, размер \(H_{out} \times W_{out}\) |
| \(\delta\) | матрица локальных градиентов, размер \(H_{out} \times W_{out}\) |
| \(\sigma\) | функция активации (в примерах — сигмоида) |

Размер выхода без учета stride и padding:
$$
H_{out} = H - K_h + 1, \qquad W_{out} = W - K_w + 1.
$$

Для нашего примера \(H=3, W=3, K_h=2, K_w=2\), значит \(H_{out} = 2, W_{out} = 2\).

Итого **пять обучаемых параметров**: четыре веса ядра \(w_{11}, w_{12}, w_{21}, w_{22}\) и одно смещение \(b\).

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

def draw_architecture_2d():
    """Рис. 3.1: Архитектура 2D-свёртки (вход 3x3, ядро 2x2, выход 2x2)."""
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_xlim(0, 14); ax.set_ylim(0, 10)
    ax.axis('off')

    # --- Входная матрица X (3x3) ---
    for i in range(3):
        for j in range(3):
            rect = patches.Rectangle((2 + j*1.2, 5 - i*1.2), 1, 1, facecolor='#AEDFF7', edgecolor='black', lw=1.5, zorder=3)
            ax.add_patch(rect)
            ax.text(2.5 + j*1.2, 5.5 - i*1.2, f'\(x_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=12, zorder=4)
    ax.text(4.0, 7.8, 'Входная матрица $X$ (3x3)', ha='center', fontsize=14, style='italic')

    # Красная рамка первого окна
    rect_window = patches.Rectangle((1.8, 4.8), 2.4, 2.4, fill=False, edgecolor='#E74C3C', linestyle='--', lw=2.5, zorder=5)
    ax.add_patch(rect_window)

    # --- Ядро W (2x2) ---
    for i in range(2):
        for j in range(2):
            rect = patches.Rectangle((6 + j*1.2, 6.5 - i*1.2), 1, 1, facecolor='#FFE29A', edgecolor='black', lw=1.5, zorder=3)
            ax.add_patch(rect)
            ax.text(6.5 + j*1.2, 7.0 - i*1.2, f'\(w_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=12, zorder=4)
    ax.text(7.0, 8.5, 'Ядро $W$ (2x2)', ha='center', fontsize=14, style='italic')

    # Стрелка от входа к ядру (прямой проход)
    ax.annotate('', xy=(5.8, 5.8), xytext=(4.2, 5.8),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
    ax.text(5.0, 6.1, 'Свёртка', ha='center', fontsize=10)

    # --- Выход Z (2x2) ---
    for i in range(2):
        for j in range(2):
            rect = patches.Rectangle((10 + j*1.5, 5 - i*1.5), 1, 1, facecolor='#E8E8E8', edgecolor='black', lw=1.5, zorder=3)
            ax.add_patch(rect)
            ax.text(10.5 + j*1.5, 5.5 - i*1.5, f'\(z_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=12, zorder=4)
    ax.text(11.2, 7.3, 'Выход $Z$ (2x2)', ha='center', fontsize=14, style='italic')

    # Смещение
    ax.annotate('', xy=(10.5, 4.8), xytext=(10.5, 3.5),
                arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5))
    ax.text(10.8, 4.0, 'Смещение \(b\)', fontsize=12, color='#1F6FB2', fontweight='bold')

    # Активация
    rect_act = patches.Rectangle((10, 1.5), 2.5, 1.2, facecolor='#B8E0B8', edgecolor='black', lw=1.5, zorder=3)
    ax.add_patch(rect_act)
    ax.text(11.25, 2.1, 'Активация \(\sigma(Z)\)', ha='center', va='center', fontsize=12, zorder=4)

    ax.text(7.0, 0.5, 'Рис. 3.1. 2D-свёртка: окно 2x2 скользит по входу 3x3',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_architecture_2d()
```

## 3.3. Прямой проход

Прямой проход — это вычисление матрицы \(Z\), а затем матрицы \(A\). Мы используем **кросс-корреляцию** (как в PyTorch/Keras), то есть ядро не переворачивается.

### Шаг 1. Линейная комбинация (преактивация)

Каждый элемент матрицы \(Z\) вычисляется как сумма произведений элементов ядра на соответствующий фрагмент входа. В общем виде:
$$
Z_{i,j} = b + \sum_{m=1}^{K_h} \sum_{n=1}^{K_w} W_{m,n} \cdot X_{i+m-1,\, j+n-1}.
$$
где \(i=1, \dots, H_{out}\) и \(j=1, \dots, W_{out}\).

Для нашего примера (\(H_{out}=2, W_{out}=2\)):
$$
Z_{1,1} = b + w_{11} x_{11} + w_{12} x_{12} + w_{21} x_{21} + w_{22} x_{22},
$$
$$
Z_{1,2} = b + w_{11} x_{12} + w_{12} x_{13} + w_{21} x_{22} + w_{22} x_{23},
$$
$$
Z_{2,1} = b + w_{11} x_{21} + w_{12} x_{22} + w_{21} x_{31} + w_{22} x_{32},
$$
$$
Z_{2,2} = b + w_{11} x_{22} + w_{12} x_{23} + w_{21} x_{32} + w_{22} x_{33}.
$$

### Шаг 2. Активация

К каждому элементу матрицы \(Z\) применяется нелинейность:
$$
A_{i,j} = \sigma(Z_{i,j}).
$$

### Шаг 3. Функция потерь

Поскольку выход — это матрица, потерю считаем как сумму квадратов ошибок по всем пикселям:
$$
L = \frac{1}{2} \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} (A_{i,j} - T_{i,j})^2.
$$

## 3.4. Обратный проход

Теперь мы применяем цепное правило. Главное отличие от 1D — **двойное суммирование** при вычислении градиентов по весам и смещению, так как каждый вес используется во всех позициях по высоте и ширине.

### Шаг 1. Локальный градиент (дельта)

Для каждого элемента матрицы выхода:
$$
\delta_{i,j} := \frac{\partial L}{\partial Z_{i,j}} = \frac{\partial L}{\partial A_{i,j}} \cdot \frac{\partial A_{i,j}}{\partial Z_{i,j}} = (A_{i,j} - T_{i,j}) \cdot \sigma'(Z_{i,j}).
$$

### Шаг 2. Градиент по смещению \(b\)

Смещение \(b\) прибавляется к **каждому** элементу \(Z_{i,j}\). По правилу суммы производных:
$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} \delta_{i,j}.
$$

### Шаг 3. Градиенты по ядру \(W_{m,n}\)

Каждый вес \(W_{m,n}\) участвует в вычислении всех \(Z_{i,j}\). Поэтому выписываем полную цепочку с двойной суммой:
$$
\frac{\partial L}{\partial W_{m,n}} = \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} \delta_{i,j} \cdot X_{i+m-1,\, j+n-1}.
$$

Распишем для каждого веса нашего примера:
$$
\frac{\partial L}{\partial w_{11}} = \delta_{1,1} x_{11} + \delta_{1,2} x_{12} + \delta_{2,1} x_{21} + \delta_{2,2} x_{22},
$$
$$
\frac{\partial L}{\partial w_{12}} = \delta_{1,1} x_{12} + \delta_{1,2} x_{13} + \delta_{2,1} x_{22} + \delta_{2,2} x_{23},
$$
$$
\frac{\partial L}{\partial w_{21}} = \delta_{1,1} x_{21} + \delta_{1,2} x_{22} + \delta_{2,1} x_{31} + \delta_{2,2} x_{32},
$$
$$
\frac{\partial L}{\partial w_{22}} = \delta_{1,1} x_{22} + \delta_{1,2} x_{23} + \delta_{2,1} x_{32} + \delta_{2,2} x_{33}.
$$

## 3.5. Обновление параметров

Все пять параметров обновляются по стандартному правилу градиентного спуска, производные берутся в текущей точке \(\theta^{(t)}\):
$$
w_{m,n}^{(t+1)} := w_{m,n}^{(t)} - \eta \left.\frac{\partial L}{\partial w_{m,n}}\right|_{\theta^{(t)}}, \qquad b^{(t+1)} := b^{(t)} - \eta \left.\frac{\partial L}{\partial b}\right|_{\theta^{(t)}}.
$$

## 3.6. Один полный шаг обучения на числах

Проверим формулы вручную. Возьмём:
- Вход: \(X = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{pmatrix}\).
- Ядро: \(W = \begin{pmatrix} 0.1 & 0.2 \\ 0.3 & 0.4 \end{pmatrix}\).
- Смещение: \(b = 0\).
- Цель: \(T = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}\).
- Шаг: \(\eta = 0.1\).

### Прямой проход

Вычисляем преактивации:
$$
Z_{1,1} = 0 + 0.1(1) + 0.2(2) + 0.3(4) + 0.4(5) = 0.1 + 0.4 + 1.2 + 2.0 = 3.7,
$$
$$
Z_{1,2} = 0 + 0.1(2) + 0.2(3) + 0.3(5) + 0.4(6) = 0.2 + 0.6 + 1.5 + 2.4 = 4.7,
$$
$$
Z_{2,1} = 0 + 0.1(4) + 0.2(5) + 0.3(7) + 0.4(8) = 0.4 + 1.0 + 2.1 + 3.2 = 6.7,
$$
$$
Z_{2,2} = 0 + 0.1(5) + 0.2(6) + 0.3(8) + 0.4(9) = 0.5 + 1.2 + 2.4 + 3.6 = 7.7.
$$

Вычисляем активации:
\(A_{1,1} = \sigma(3.7) \approx 0.97574\),
\(A_{1,2} = \sigma(4.7) \approx 0.99099\),
\(A_{2,1} = \sigma(6.7) \approx 0.99878\),
\(A_{2,2} = \sigma(7.7) \approx 0.99955\).

### Обратный проход

Дельты:
$$
\delta_{1,1} = (0.97574 - 1) \cdot 0.97574(1-0.97574) \approx (-0.02426)(0.02367) \approx -0.000574,
$$
$$
\delta_{1,2} = (0.99099 - 1) \cdot 0.99099(1-0.99099) \approx (-0.00901)(0.00893) \approx -0.000080,
$$
$$
\delta_{2,1} = (0.99878 - 1) \cdot 0.99878(1-0.99878) \approx (-0.00122)(0.00122) \approx -0.0000015,
$$
$$
\delta_{2,2} = (0.99955 - 1) \cdot 0.99955(1-0.99955) \approx (-0.00045)(0.00045) \approx -0.0000002.
$$

*(Заметим, что из-за сигмоиды при больших \(Z\) градиенты очень малы. Это нормально).*

Градиент по смещению:
$$
\frac{\partial L}{\partial b} = -0.000574 - 0.000080 - 0.0000015 - 0.0000002 \approx -0.000655.
$$

Градиенты по весам:
$$
\frac{\partial L}{\partial w_{11}} = (-0.000574)(1) + (-0.000080)(2) + (-0.0000015)(4) + (-0.0000002)(5) \approx -0.000743,
$$
$$
\frac{\partial L}{\partial w_{12}} = (-0.000574)(2) + (-0.000080)(3) + (-0.0000015)(5) + (-0.0000002)(6) \approx -0.001401,
$$
$$
\frac{\partial L}{\partial w_{21}} = (-0.000574)(4) + (-0.000080)(5) + (-0.0000015)(7) + (-0.0000002)(8) \approx -0.002710,
$$
$$
\frac{\partial L}{\partial w_{22}} = (-0.000574)(5) + (-0.000080)(6) + (-0.0000015)(8) + (-0.0000002)(9) \approx -0.003369.
$$

Обновление (шаг \(\eta = 0.1\)):
$$
w_{11} \leftarrow 0.1 - 0.1(-0.000743) = 0.100074,
$$
$$
w_{12} \leftarrow 0.2 - 0.1(-0.001401) = 0.200140,
$$
$$
w_{21} \leftarrow 0.3 - 0.1(-0.002710) = 0.300271,
$$
$$
w_{22} \leftarrow 0.4 - 0.1(-0.003369) = 0.400337,
$$
$$
b \leftarrow 0 - 0.1(-0.000655) = 0.0000655.
$$

## 3.7. Полная программа

```python
import numpy as np
import math

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1 - s)

class TinyConv2D:
    """Свёртка 2D: один канал, одно ядро, stride=1, padding=0."""
    def __init__(self, W, b):
        self.W = np.array(W)  # размер (Kh, Kw)
        self.b = float(b)

    def forward(self, X):
        H, W_in = X.shape
        Kh, Kw = self.W.shape
        H_out = H - Kh + 1
        W_out = W_in - Kw + 1
        
        Z = np.zeros((H_out, W_out))
        A = np.zeros((H_out, W_out))
        
        for i in range(H_out):
            for j in range(W_out):
                # Вырезаем окно
                window = X[i:i+Kh, j:j+Kw]
                # Скалярное произведение
                Z[i, j] = self.b + np.sum(window * self.W)
                A[i, j] = sigmoid(Z[i, j])
        return Z, A

    def backward(self, X, T, Z, A):
        H, W_in = X.shape
        Kh, Kw = self.W.shape
        H_out, W_out = Z.shape
        
        delta = (A - T) * sigmoid_prime(Z)
        
        db = np.sum(delta)
        
        dW = np.zeros_like(self.W)
        for m in range(Kh):
            for n in range(Kw):
                for i in range(H_out):
                    for j in range(W_out):
                        dW[m, n] += delta[i, j] * X[i+m, j+n]
                        
        return dW, db

    def update(self, dW, db, eta):
        self.W -= eta * dW
        self.b -= eta * db

# Демонстрация
if __name__ == '__main__':
    X = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
    T = np.array([[1, 1], [1, 1]], dtype=float)
    net = TinyConv2D(W=[[0.1, 0.2], [0.3, 0.4]], b=0.0)
    
    Z, A = net.forward(X)
    dW, db = net.backward(X, T, Z, A)
    
    print("Прямой проход:")
    print("Z =\n", Z)
    print("A =\n", A)
    
    print("\nОбратный проход:")
    print("dW =\n", dW)
    print("db =", db)
```

## 3.8. Что мы получили

В этой главе мы разобрали двумерную свёртку:

1. **Прямой проход** — это скользящее окно \(K_h \times K_w\) по матрице \(H \times W\). Выход — матрица \(H_{out} \times W_{out}\).
2. **Обратный проход** — это **двойное суммирование** по всем позициям \((i, j)\). Каждый вес \(w_{m,n}\) собирает градиенты со всех мест, куда он «приложился».
3. **Размерности** — теперь мы работаем с матрицами. Градиент по смещению \(b\) — скаляр (сумма всех \(\delta\)), градиент по ядру — матрица того же размера, что и само ядро.

Этот механизм — строительный блок для любой современной CNN. В следующей главе мы добавим **многоканальность** (когда у нас не 1, а, скажем, 3 входных канала — RGB, и несколько ядер), и вы увидите, как формулы обобщаются на тензоры.


# Глава 4. Пример 4: Многоканальная 2D-свёртка и ReLU

В Главе 3 мы рассмотрели 2D-свёртку для одного входного канала. Но реальные изображения всегда имеют глубину. Даже черно-белые картинки в современных сетях на входе часто имеют несколько каналов. Если мы хотим, чтобы сеть улавливала сложные признаки (грани, текстуры, цвета), нам нужно использовать **многоканальные ядра**. Кроме того, мы заменим сигмоиду на **ReLU**, чтобы градиенты не исчезали в глубоких сетях.

## 4.1. Зачем нужен этот пример

Многоканальная свёртка — это сердце любого современного зрения (ResNet, YOLO, EfficientNet). Она позволяет сети «смотреть» на разные цветовые каналы одновременно.
Главные отличия от Главы 3:
- **Вход:** Не матрица \(H \times W\), а тензор \(C_{in} \times H \times W\) (где \(C_{in}\) — количество входных каналов).
- **Ядро:** Не матрица \(K_h \times K_w\), а тензор \(C_{in} \times K_h \times K_w\).
- **Операция:** Каждый канал входа сворачивается со своим слоем ядра, а результаты **суммируются** вместе с добавлением смещения.
- **Активация:** Вместо сигмоиды используем ReLU: \(f(x) = \max(0, x)\).

## 4.2. Архитектура и обозначения (Рис. 4.1)

Для простоты и наглядности мы возьмем:
- **Вход:** \(C_{in} = 2\) канала, каждый размером \(3 \times 3\).
- **Ядро:** 1 фильтр, состоящий из 2 слоев (по одному на каждый входной канал), размер каждого слоя \(2 \times 2\).
- **Выход:** 1 канал размером \(2 \times 2\).

| Символ | Смысл |
|---|---|
| \(X\) | входной тензор, размер \(C_{in} \times H \times W\) |
| \(W\) | ядро, размер \(C_{in} \times K_h \times K_w\) |
| \(b\) | смещение (bias) |
| \(Z\) | матрица преактиваций, размер \(H_{out} \times W_{out}\) |
| \(A\) | матрица активаций (выход), размер \(H_{out} \times W_{out}\) |
| \(T\) | матрица истинных ответов |
| \(\delta\) | матрица локальных градиентов |
| \(\sigma\) | функция активации ReLU |

Размер выхода: \(H_{out} = H - K_h + 1 = 3 - 2 + 1 = 2\), аналогично \(W_{out} = 2\).

Итого **девять обучаемых параметров**: восемь весов (по 4 на каждый канал ядра) и одно смещение \(b\).

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_architecture_multichannel():
    """Рис. 4.1: Многоканальная 2D-свёртка (2 канала -> 1 фильтр -> 1 канал)."""
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 16); ax.set_ylim(0, 10)
    ax.axis('off')

    # --- Входной тензор (2 канала по 3x3) ---
    ax.text(3.0, 9.0, 'Вход $X$ (2 канала, 3x3)', ha='center', fontsize=14, style='italic')
    
    # Канал 1
    for i in range(3):
        for j in range(3):
            rect = patches.Rectangle((1 + j*1.0, 6.5 - i*1.0), 0.9, 0.9, facecolor='#AEDFF7', edgecolor='black', lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.text(1.45 + j*1.0, 6.95 - i*1.0, f'\(x^1_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10, zorder=4)
    ax.text(2.0, 5.2, 'Канал 1', ha='center', fontsize=11, color='#1F6FB2')

    # Канал 2 (смещен вправо)
    for i in range(3):
        for j in range(3):
            rect = patches.Rectangle((3.5 + j*1.0, 6.5 - i*1.0), 0.9, 0.9, facecolor='#AEDFF7', edgecolor='black', lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.text(3.95 + j*1.0, 6.95 - i*1.0, f'\(x^2_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10, zorder=4)
    ax.text(4.5, 5.2, 'Канал 2', ha='center', fontsize=11, color='#1F6FB2')

    # --- Ядро (2 канала по 2x2) ---
    ax.text(9.5, 9.0, 'Ядро $W$ (2 канала, 2x2)', ha='center', fontsize=14, style='italic')
    
    # Слой ядра 1
    for i in range(2):
        for j in range(2):
            rect = patches.Rectangle((8 + j*1.0, 6.5 - i*1.0), 0.9, 0.9, facecolor='#FFE29A', edgecolor='black', lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.text(8.45 + j*1.0, 6.95 - i*1.0, f'\(w^1_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10, zorder=4)

    # Слой ядра 2
    for i in range(2):
        for j in range(2):
            rect = patches.Rectangle((10.5 + j*1.0, 6.5 - i*1.0), 0.9, 0.9, facecolor='#FFE29A', edgecolor='black', lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.text(10.95 + j*1.0, 6.95 - i*1.0, f'\(w^2_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10, zorder=4)

    # --- Стрелки (свёртка) ---
    ax.annotate('', xy=(7.5, 6.0), xytext=(5.0, 6.0), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
    ax.annotate('', xy=(7.5, 6.0), xytext=(5.0, 6.0), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
    ax.text(6.25, 6.3, 'Свёртка\n(сумма по каналам)', ha='center', fontsize=10)

    # --- Выход (1 канал 2x2) ---
    ax.text(13.0, 7.5, 'Выход $Z$ (2x2)', ha='center', fontsize=14, style='italic')
    for i in range(2):
        for j in range(2):
            rect = patches.Rectangle((12.5 + j*1.2, 5.5 - i*1.2), 1.0, 1.0, facecolor='#E8E8E8', edgecolor='black', lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.text(13.0 + j*1.2, 6.0 - i*1.2, f'\(z_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=12, zorder=4)

    # Смещение
    ax.annotate('', xy=(13.5, 3.0), xytext=(13.5, 1.5), arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5))
    ax.text(14.0, 2.2, 'Смещение \(b\)', fontsize=12, color='#1F6FB2', fontweight='bold')

    # Активация
    rect_act = patches.Rectangle((12.0, 0.0), 3.0, 1.2, facecolor='#B8E0B8', edgecolor='black', lw=1.5, zorder=3)
    ax.add_patch(rect_act)
    ax.text(13.5, 0.6, 'Активация ReLU', ha='center', va='center', fontsize=12, zorder=4)

    ax.text(8.0, -0.8, 'Рис. 4.1. Многоканальная свёртка: каждый канал входа перемножается со своим слоем ядра, результаты суммируются.',
            ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_architecture_multichannel()
```

## 4.3. Прямой проход

Прямой проход теперь включает **суммирование по каналам**.

### Шаг 1. Линейная комбинация (преактивация)

Чтобы получить элемент \(Z_{i,j}\), мы должны сложить произведения по всем входным каналам \(c\), а также по высоте \(m\) и ширине \(n\):
$$
Z_{i,j} = b + \sum_{c=1}^{C_{in}} \sum_{m=1}^{K_h} \sum_{n=1}^{K_w} W_{c,m,n} \cdot X_{c,\,i+m-1,\,j+n-1}.
$$
Для нашего примера (\(C_{in}=2\), \(K_h=K_w=2\)):
$$
Z_{1,1} = b + w^1_{11} x^1_{11} + w^1_{12} x^1_{12} + w^1_{21} x^1_{21} + w^1_{22} x^1_{22} + w^2_{11} x^2_{11} + w^2_{12} x^2_{12} + w^2_{21} x^2_{21} + w^2_{22} x^2_{22}.
$$
*(Обратите внимание: мы просто складываем результаты свёртки первого канала и второго канала, плюс смещение).*

### Шаг 2. Активация ReLU

Вместо сигмоиды используем ReLU. Она поэлементно обнуляет все отрицательные значения:
$$
A_{i,j} = \text{ReLU}(Z_{i,j}) = \max(0, Z_{i,j}).
$$
*Почему ReLU лучше?* Её производная равна \(1\) для \(Z > 0\) и \(0\) для \(Z < 0\). Это позволяет ошибке проходить через слои без затухания (в отличие от сигмоиды, производная которой \(\le 0.25\)).

### Шаг 3. Функция потерь

$$
L = \frac{1}{2} \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} (A_{i,j} - T_{i,j})^2.
$$

## 4.4. Обратный проход

Мы снова применяем цепное правило. Но теперь нам нужно аккуратно распределить ошибку по всем каналам.

### Шаг 1. Локальный градиент (дельта)

Производная ReLU:
$$
\text{ReLU}'(Z_{i,j}) = \begin{cases} 1, & \text{если } Z_{i,j} > 0 \\ 0, & \text{если } Z_{i,j} \le 0 \end{cases}
$$
Тогда дельта:
$$
\delta_{i,j} = \frac{\partial L}{\partial Z_{i,j}} = (A_{i,j} - T_{i,j}) \cdot \text{ReLU}'(Z_{i,j}).
$$

### Шаг 2. Градиент по смещению \(b\)

Смещение прибавляется один раз к каждому \(Z_{i,j}\), поэтому суммируем дельты по всем позициям:
$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} \delta_{i,j}.
$$

### Шаг 3. Градиенты по весам ядра \(W_{c,m,n}\)

Теперь каждый вес принадлежит конкретному каналу \(c\). Вес \(w^c_{m,n}\) взаимодействует только с элементами \(c\)-го канала входа. Поэтому полная цепочка:
$$
\frac{\partial L}{\partial W_{c,m,n}} = \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} \delta_{i,j} \cdot X_{c,\, i+m-1,\, j+n-1}.
$$
Расписываем для первого канала (\(c=1\)):
$$
\frac{\partial L}{\partial w^1_{11}} = \sum_{i,j} \delta_{i,j} x^1_{i,j}, \quad
\frac{\partial L}{\partial w^1_{12}} = \sum_{i,j} \delta_{i,j} x^1_{i,j+1},
$$
$$
\frac{\partial L}{\partial w^1_{21}} = \sum_{i,j} \delta_{i,j} x^1_{i+1,j}, \quad
\frac{\partial L}{\partial w^1_{22}} = \sum_{i,j} \delta_{i,j} x^1_{i+1,j+1}.
$$
Аналогично для второго канала (\(c=2\)), только с использованием \(x^2\).

### Шаг 4. Градиент по входу \(X\) (для предыдущих слоев)

Если бы это был не первый слой сети, нам нужно было бы передать ошибку назад на вход. Каждый канал \(c\) получает свой градиент от ядра \(c\):
$$
\frac{\partial L}{\partial X_{c, i, j}} = \sum_{m,n} \delta_{i-m+1, j-n+1} \cdot W_{c,m,n}.
$$

## 4.5. Обновление параметров

Стандартный шаг градиентного спуска:
$$
W_{c,m,n}^{(t+1)} := W_{c,m,n}^{(t)} - \eta \left.\frac{\partial L}{\partial W_{c,m,n}}\right|_{\theta^{(t)}}, \qquad b^{(t+1)} := b^{(t)} - \eta \left.\frac{\partial L}{\partial b}\right|_{\theta^{(t)}}.
$$

## 4.6. Один полный шаг обучения на числах

Для простоты и наглядности возьмем:
- Вход \(X\) (два канала):
$$
X_1 = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}, \quad X_2 = \begin{pmatrix} 0 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 1 & 0 \end{pmatrix}.
$$
- Ядро \(W\) (два канала):
$$
W_1 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad W_2 = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}.
$$
- Смещение \(b = -1\).
- Цель \(T = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}\) (идеальный выход).
- Шаг \(\eta = 0.1\).

### Прямой проход

$$
Z_{1,1} = -1 + (1\cdot1 + 0\cdot0 + 0\cdot0 + 1\cdot1) + (0\cdot0 + 1\cdot1 + 1\cdot1 + 0\cdot0) = -1 + 2 + 2 = 3.
$$
$$
Z_{1,2} = -1 + (1\cdot0 + 0\cdot0 + 0\cdot1 + 1\cdot0) + (0\cdot1 + 1\cdot0 + 1\cdot1 + 0\cdot1) = -1 + 0 + 1 = 0.
$$
$$
Z_{2,1} = -1 + (1\cdot0 + 0\cdot0 + 0\cdot0 + 1\cdot0) + (0\cdot1 + 1\cdot0 + 1\cdot0 + 0\cdot1) = -1 + 0 + 1 = 0.
$$
$$
Z_{2,2} = -1 + (1\cdot1 + 0\cdot0 + 0\cdot0 + 1\cdot1) + (0\cdot0 + 1\cdot1 + 1\cdot1 + 0\cdot0) = -1 + 2 + 2 = 3.
$$
Итак, \(Z = \begin{pmatrix} 3 & 0 \\ 0 & 3 \end{pmatrix}\).

Применяем ReLU:
$$
A = \text{ReLU}(Z) = \begin{pmatrix} 3 & 0 \\ 0 & 3 \end{pmatrix}.
$$
*(Все значения положительные или нулевые).*

Потеря \(L\): \((3-1)^2 + (0-1)^2 + (0-1)^2 + (3-1)^2 = 4 + 1 + 1 + 4 = 10\).

### Обратный проход

Дельты (производная ReLU для \(Z>0\) равна 1, для \(Z=0\) равна 0, но мы берем \(Z=0\) как границу, где градиент 0):
$$
\delta_{1,1} = (3 - 1) \cdot 1 = 2,
$$
$$
\delta_{1,2} = (0 - 1) \cdot 0 = 0,
$$
$$
\delta_{2,1} = (0 - 1) \cdot 0 = 0,
$$
$$
\delta_{2,2} = (3 - 1) \cdot 1 = 2.
$$
Итак, \(\delta = \begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix}\).

Градиент по смещению:
$$
\frac{\partial L}{\partial b} = 2 + 0 + 0 + 2 = 4.
$$

Градиенты по весам первого канала (\(W_1\)):
$$
\frac{\partial L}{\partial w^1_{11}} = \delta_{1,1} x^1_{11} + \delta_{1,2} x^1_{12} + \delta_{2,1} x^1_{21} + \delta_{2,2} x^1_{22} = 2(1) + 0(0) + 0(0) + 2(1) = 4,
$$
$$
\frac{\partial L}{\partial w^1_{12}} = \delta_{1,1} x^1_{12} + \delta_{1,2} x^1_{13} + \delta_{2,1} x^1_{22} + \delta_{2,2} x^1_{23} = 2(0) + 0(0) + 0(1) + 2(0) = 0,
$$
$$
\frac{\partial L}{\partial w^1_{21}} = \delta_{1,1} x^1_{21} + \delta_{1,2} x^1_{22} + \delta_{2,1} x^1_{31} + \delta_{2,2} x^1_{32} = 2(0) + 0(1) + 0(0) + 2(0) = 0,
$$
$$
\frac{\partial L}{\partial w^1_{22}} = \delta_{1,1} x^1_{22} + \delta_{1,2} x^1_{23} + \delta_{2,1} x^1_{32} + \delta_{2,2} x^1_{33} = 2(1) + 0(0) + 0(0) + 2(1) = 4.
$$
То есть \(dW_1 = \begin{pmatrix} 4 & 0 \\ 0 & 4 \end{pmatrix}\).

Градиенты по весам второго канала (\(W_2\)):
$$
\frac{\partial L}{\partial w^2_{11}} = 2(0) + 0(1) + 0(1) + 2(0) = 0,
$$
$$
\frac{\partial L}{\partial w^2_{12}} = 2(1) + 0(0) + 0(0) + 2(1) = 4,
$$
$$
\frac{\partial L}{\partial w^2_{21}} = 2(1) + 0(0) + 0(0) + 2(1) = 4,
$$
$$
\frac{\partial L}{\partial w^2_{22}} = 2(0) + 0(1) + 0(1) + 2(0) = 0.
$$
То есть \(dW_2 = \begin{pmatrix} 0 & 4 \\ 4 & 0 \end{pmatrix}\).

### Обновление (шаг \(\eta = 0.1\))
$$
W_1 \leftarrow \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} - 0.1 \begin{pmatrix} 4 & 0 \\ 0 & 4 \end{pmatrix} = \begin{pmatrix} 0.6 & 0 \\ 0 & 0.6 \end{pmatrix},
$$
$$
W_2 \leftarrow \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} - 0.1 \begin{pmatrix} 0 & 4 \\ 4 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 0.6 \\ 0.6 & 0 \end{pmatrix},
$$
$$
b \leftarrow -1 - 0.1(4) = -1.4.
$$
*(Заметим, что веса, которые "смотрели" на правильные пиксели, уменьшились, чтобы сбить слишком высокий \(Z=3\) ближе к \(T=1\)).*

## 4.7. Полная программа

```python
import numpy as np

def relu(z):
    return np.maximum(0, z)

def relu_prime(z):
    return np.where(z > 0, 1.0, 0.0)

class MultiChannelConv2D:
    """Многоканальная 2D-свёртка (1 фильтр, stride=1, padding=0)."""
    def __init__(self, W, b):
        # W имеет размерность (C_in, Kh, Kw)
        self.W = np.array(W)
        self.b = float(b)

    def forward(self, X):
        # X имеет размерность (C_in, H, W)
        C_in, H, W_in = X.shape
        _, Kh, Kw = self.W.shape
        H_out = H - Kh + 1
        W_out = W_in - Kw + 1
        
        Z = np.zeros((H_out, W_out))
        A = np.zeros((H_out, W_out))
        
        for i in range(H_out):
            for j in range(W_out):
                # Вырезаем окно по всем каналам
                window = X[:, i:i+Kh, j:j+Kw]
                # Сумма произведений по всем каналам и пространству
                Z[i, j] = self.b + np.sum(window * self.W)
                A[i, j] = relu(Z[i, j])
        return Z, A

    def backward(self, X, T, Z, A):
        C_in, H, W_in = X.shape
        _, Kh, Kw = self.W.shape
        H_out, W_out = Z.shape
        
        delta = (A - T) * relu_prime(Z)
        
        db = np.sum(delta)
        
        dW = np.zeros_like(self.W)
        for c in range(C_in):
            for m in range(Kh):
                for n in range(Kw):
                    for i in range(H_out):
                        for j in range(W_out):
                            dW[c, m, n] += delta[i, j] * X[c, i+m, j+n]
                            
        return dW, db

    def update(self, dW, db, eta):
        self.W -= eta * dW
        self.b -= eta * db

# Демонстрация
if __name__ == '__main__':
    X = np.array([
        [[1, 0, 0], [0, 1, 0], [0, 0, 1]],  # Канал 1
        [[0, 1, 0], [1, 0, 1], [0, 1, 0]]   # Канал 2
    ], dtype=float)
    T = np.ones((2, 2))
    
    W = np.array([
        [[1, 0], [0, 1]],  # Канал 1
        [[0, 1], [1, 0]]   # Канал 2
    ], dtype=float)
    
    net = MultiChannelConv2D(W=W, b=-1.0)
    
    Z, A = net.forward(X)
    dW, db = net.backward(X, T, Z, A)
    
    print("Z =\n", Z)
    print("A =\n", A)
    print("dW =\n", dW)
    print("db =", db)
    
    net.update(dW, db, eta=0.1)
    print("W после обновления =\n", net.W)
    print("b после обновления =", net.b)
```

## 4.8. Что мы получили

В этой главе мы сделали огромный шаг к реальным нейронным сетям:

1. **Тензоры 3D:** Мы научились обрабатывать входы с несколькими каналами. Каждый канал имеет свой слой в ядре, и результаты суммируются в один выход.
2. **ReLU:** Заменив сигмоиду на ReLU, мы избавились от проблемы затухания градиента. Производная ReLU — это просто маска из нулей и единиц, что делает вычисления очень быстрыми.
3. **Суммирование по каналам:** В обратном проходе градиент ошибки \(\delta\) распределяется по всем каналам ядра. Вес \(w^1_{11}\) получает градиент от пикселей первого канала, а \(w^2_{11}\) — от второго.
4. **Фундамент CNN:** Теперь мы знаем, как устроен базовый слой `nn.Conv2d`. В реальных сетях добавляется только **множество фильтров** (когда \(C_{out} > 1\), мы просто повторяем этот механизм для каждого выходного фильтра).

Дальше мы можем разобрать **Pooling (MaxPool/AvgPool)**, чтобы понять, как сети уменьшают размерность картинки, или **Flatten**, чтобы перейти от свёрток к полносвязным слоям (которые вы уже знаете).



# Глава 5. Пример 5: Pooling (MaxPool и AvgPool)

До этого мы рассматривали слои, которые имеют обучаемые параметры (веса \(W\) и смещения \(b\)). Pooling-слои (слои подвыборки) **не имеют обучаемых параметров**. Их задача — агрегировать информацию в небольшом окне, уменьшая пространственные размеры (высоту и ширину) карты признаков.

Мы разберем два самых популярных типа:
1. **MaxPool** — выбирает максимальное значение в окне.
2. **AvgPool** — вычисляет среднее значение в окне.

## 5.1. Зачем нужен этот пример

В реальных CNN после свёрток всегда ставят Pooling. Он позволяет уменьшить размерность, добавить инвариантность к сдвигу и увеличить рецептивное поле.

Поскольку в Pooling-слоях нет параметров, при обратном проходе нам **не нужно** обновлять веса. Наша единственная задача — вычислить градиент по входу \(\frac{\partial L}{\partial X}\) и передать его на предыдущий слой (например, на свёртку), чтобы там обновились веса.

## 5.2. Архитектура и обозначения (Рис. 5.1)

Возьмем входную матрицу \(X\) размером \(H \times W = 4 \times 4\) (один канал).
Окно (kernel size) \(K = 2\).
Шаг (stride) \(s = 2\).
Padding = 0.

| Символ | Смысл |
|---|---|
| \(X_{p,q}\) | элемент входной матрицы, \(p=1..H, q=1..W\) |
| \(Z_{i,j}\) | элемент выхода, \(i=1..H_{out}, j=1..W_{out}\) |
| \(H_{out}\) | размер выхода по высоте: \(\lfloor (H - K)/s \rfloor + 1\) |
| \(W_{out}\) | размер выхода по ширине: \(\lfloor (W - K)/s \rfloor + 1\) |
| \(\delta_{i,j}\) | градиент от следующего слоя: \(\delta_{i,j} = \frac{\partial L}{\partial Z_{i,j}}\) |
| \(\frac{\partial L}{\partial X_{p,q}}\) | градиент по входу, который мы хотим найти |

Для нашего примера \(H_{out} = W_{out} = 2\).

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_pooling_architecture():
    """Рис. 5.1: Схема Pooling 2x2 со Stride=2."""
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(0, 14); ax.set_ylim(0, 7)
    ax.axis('off')

    # --- Входная матрица 4x4 ---
    for i in range(4):
        for j in range(4):
            rect = patches.Rectangle((1 + j*1.0, 5 - i*1.0), 0.9, 0.9, facecolor='#AEDFF7', edgecolor='black', lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.text(1.45 + j*1.0, 5.45 - i*1.0, f'{i*4+j+1}', ha='center', va='center', fontsize=12, zorder=4)
    ax.text(2.5, 1.2, 'Вход \(X\) (4x4)', ha='center', fontsize=12, style='italic')

    # Красные рамки для окон (Stride=2)
    rect_win1 = patches.Rectangle((0.8, 4.1), 2.2, 2.2, fill=False, edgecolor='#E74C3C', linestyle='--', lw=2.5, zorder=5)
    rect_win2 = patches.Rectangle((2.8, 4.1), 2.2, 2.2, fill=False, edgecolor='#2980B9', linestyle='--', lw=2.5, zorder=5)
    rect_win3 = patches.Rectangle((0.8, 2.1), 2.2, 2.2, fill=False, edgecolor='#27AE60', linestyle='--', lw=2.5, zorder=5)
    rect_win4 = patches.Rectangle((2.8, 2.1), 2.2, 2.2, fill=False, edgecolor='#8E44AD', linestyle='--', lw=2.5, zorder=5)
    ax.add_patch(rect_win1); ax.add_patch(rect_win2); ax.add_patch(rect_win3); ax.add_patch(rect_win4)

    # --- Стрелки ---
    ax.annotate('', xy=(7.5, 3.5), xytext=(5.5, 3.5), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
    ax.text(6.5, 3.8, 'Pooling', ha='center', fontsize=11)

    # --- Выходная матрица 2x2 ---
    for i in range(2):
        for j in range(2):
            rect = patches.Rectangle((8.5 + j*1.5, 4 - i*1.5), 1.2, 1.2, facecolor='#E8E8E8', edgecolor='black', lw=1.5, zorder=3)
            ax.add_patch(rect)
            ax.text(9.1 + j*1.5, 4.6 - i*1.5, f'\(Z_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=14, zorder=4)
    
    ax.text(9.85, 1.2, 'Выход \(Z\) (2x2)', ha='center', fontsize=12, style='italic')

    ax.text(7.0, 6.5, 'Рис. 5.1. Pooling окно 2x2, Stride 2: матрица 4x4 сжимается в 2x2',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_pooling_architecture()
```

## 5.3. MaxPool: Прямой проход

MaxPool выбирает максимальное значение в каждом окне. В общем виде:
$$
Z_{i,j} = \max_{m=1..K, \, n=1..K} X_{i \cdot s + m, \, j \cdot s + n}.
$$
Для каждого окна мы также запоминаем **индекс** (позицию), где этот максимум находился:
$$
(m^*, n^*) = \arg\max_{m,n} X_{i \cdot s + m, \, j \cdot s + n}.
$$
Этот индекс критически важен для обратного прохода.

## 5.4. MaxPool: Обратный проход

Нам нужно найти \(\frac{\partial L}{\partial X_{p,q}}\). Применяем цепное правило:
$$
\frac{\partial L}{\partial X_{p,q}} = \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} \frac{\partial L}{\partial Z_{i,j}} \cdot \frac{\partial Z_{i,j}}{\partial X_{p,q}}.
$$
Поскольку \(Z_{i,j}\) — это максимум, его производная по аргументам равна 1 только для того элемента, который победил, и 0 для всех остальных. Формально:
$$
\frac{\partial Z_{i,j}}{\partial X_{p,q}} = \begin{cases} 1, & \text{если } p = i \cdot s + m^* \text{ и } q = j \cdot s + n^* \\ 0, & \text{иначе} \end{cases}
$$
Тогда вся сумма схлопывается в одно слагаемое. Градиент \(\delta_{i,j}\) передается **только** тому элементу входа, который был максимумом:
$$
\frac{\partial L}{\partial X_{p,q}} = \delta_{i,j}, \quad \text{где } (p,q) = (i \cdot s + m^*, \, j \cdot s + n^*).
$$
Для всех остальных \(X_{p,q}\) в окне градиент равен \(0\).

## 5.5. AvgPool: Прямой проход

AvgPool вычисляет среднее арифметическое всех значений в окне:
$$
Z_{i,j} = \frac{1}{K^2} \sum_{m=1}^{K} \sum_{n=1}^{K} X_{i \cdot s + m, \, j \cdot s + n}.
$$

## 5.6. AvgPool: Обратный проход

Снова применяем цепное правило:
$$
\frac{\partial L}{\partial X_{p,q}} = \sum_{i=1}^{H_{out}} \sum_{j=1}^{W_{out}} \frac{\partial L}{\partial Z_{i,j}} \cdot \frac{\partial Z_{i,j}}{\partial X_{p,q}}.
$$
Среднее — это линейная комбинация. Производная \(Z_{i,j}\) по любому \(X_{p,q}\) внутри окна равна константе:
$$
\frac{\partial Z_{i,j}}{\partial X_{p,q}} = \frac{1}{K^2}, \quad \text{для всех } X_{p,q} \text{ в окне } (i,j).
$$
Следовательно, ошибка \(\delta_{i,j}\) **равномерно распределяется** между всеми элементами окна:
$$
\frac{\partial L}{\partial X_{p,q}} = \sum_{(i,j) \text{ накрывающих } (p,q)} \delta_{i,j} \cdot \frac{1}{K^2}.
$$

## 5.7. Один полный шаг обучения на числах

Возьмем входную матрицу \(X\) размером \(4 \times 4\):
$$
X = \begin{pmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
9 & 10 & 11 & 12 \\
13 & 14 & 15 & 16
\end{pmatrix}.
$$
Пусть градиент от следующего слоя:
$$
\delta = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}.
$$

### MaxPool

**Прямой проход:**
- Окно 1 (верхний левый): \(\max(1,2,5,6) = 6\). Индекс максимума: \(p=2, q=2\).
- Окно 2 (верхний правый): \(\max(3,4,7,8) = 8\). Индекс: \(p=2, q=4\).
- Окно 3 (нижний левый): \(\max(9,10,13,14) = 14\). Индекс: \(p=4, q=2\).
- Окно 4 (нижний правый): \(\max(11,12,15,16) = 16\). Индекс: \(p=4, q=4\).

$$
Z = \begin{pmatrix} 6 & 8 \\ 14 & 16 \end{pmatrix}.
$$

**Обратный проход:**
Согласно формуле, \(\frac{\partial L}{\partial X_{p,q}}\) равно \(\delta_{i,j}\) только в позициях максимумов, остальные — нули:
$$
\frac{\partial L}{\partial X} = \begin{pmatrix}
0 & 0 & 0 & 0 \\
0 & 1 & 0 & 1 \\
0 & 0 & 0 & 0 \\
0 & 1 & 0 & 1
\end{pmatrix}.
$$
*(Например, \(\frac{\partial L}{\partial X_{2,2}} = \delta_{1,1} = 1\), а \(\frac{\partial L}{\partial X_{1,1}} = 0\).)*

### AvgPool

**Прямой проход:**
- Окно 1: \((1+2+5+6)/4 = 3.5\).
- Окно 2: \((3+4+7+8)/4 = 5.5\).
- Окно 3: \((9+10+13+14)/4 = 11.5\).
- Окно 4: \((11+12+15+16)/4 = 13.5\).

$$
Z = \begin{pmatrix} 3.5 & 5.5 \\ 11.5 & 13.5 \end{pmatrix}.
$$

**Обратный проход:**
Каждый элемент в окне получает \(\delta_{i,j} / K^2 = \delta_{i,j} / 4\). Так как все \(\delta_{i,j} = 1\):
$$
\frac{\partial L}{\partial X} = \begin{pmatrix}
0.25 & 0.25 & 0.25 & 0.25 \\
0.25 & 0.25 & 0.25 & 0.25 \\
0.25 & 0.25 & 0.25 & 0.25 \\
0.25 & 0.25 & 0.25 & 0.25
\end{pmatrix}.
$$

## 5.8. Полная программа

```python
import numpy as np

class MaxPool2D:
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, X):
        H, W = X.shape
        K = self.pool_size
        S = self.stride
        H_out = (H - K) // S + 1
        W_out = (W - K) // S + 1
        
        Z = np.zeros((H_out, W_out))
        self.indices = np.zeros((H_out, W_out), dtype=object)
        
        for i in range(H_out):
            for j in range(W_out):
                start_i = i * S
                start_j = j * S
                window = X[start_i:start_i+K, start_j:start_j+K]
                
                max_val = np.max(window)
                max_idx_flat = np.argmax(window)
                row, col = np.unravel_index(max_idx_flat, window.shape)
                
                Z[i, j] = max_val
                self.indices[i, j] = (start_i + row, start_j + col)
                
        return Z

    def backward(self, delta, X_shape):
        H, W = X_shape
        dX = np.zeros((H, W))
        
        for i in range(delta.shape[0]):
            for j in range(delta.shape[1]):
                row, col = self.indices[i, j]
                dX[row, col] += delta[i, j]
                
        return dX


class AvgPool2D:
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, X):
        H, W = X.shape
        K = self.pool_size
        S = self.stride
        H_out = (H - K) // S + 1
        W_out = (W - K) // S + 1
        
        Z = np.zeros((H_out, W_out))
        
        for i in range(H_out):
            for j in range(W_out):
                start_i = i * S
                start_j = j * S
                window = X[start_i:start_i+K, start_j:start_j+K]
                Z[i, j] = np.mean(window)
                
        return Z

    def backward(self, delta, X_shape):
        H, W = X_shape
        K = self.pool_size
        S = self.stride
        dX = np.zeros((H, W))
        
        for i in range(delta.shape[0]):
            for j in range(delta.shape[1]):
                start_i = i * S
                start_j = j * S
                dX[start_i:start_i+K, start_j:start_j+K] += delta[i, j] / (K * K)
                
        return dX

if __name__ == '__main__':
    X = np.array([
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ])
    delta = np.array([[1, 1], [1, 1]])
    
    print("=== MaxPool ===")
    max_pool = MaxPool2D()
    Z_max = max_pool.forward(X)
    dX_max = max_pool.backward(delta, X.shape)
    print("Z_max =\n", Z_max)
    print("dX_max =\n", dX_max)
    
    print("\n=== AvgPool ===")
    avg_pool = AvgPool2D()
    Z_avg = avg_pool.forward(X)
    dX_avg = avg_pool.backward(delta, X.shape)
    print("Z_avg =\n", Z_avg)
    print("dX_avg =\n", dX_avg)
```

## 5.9. Что мы получили

В этой главе мы разобрали Pooling-слои:

1. **MaxPool** в прямом проходе выбирает максимум и **запоминает его позицию**. В обратном проходе градиент \(\delta_{i,j}\) идет **только** в один победивший пиксель. Формула: \(\frac{\partial L}{\partial X_{p,q}} = \delta_{i,j}\) для позиции максимума.
2. **AvgPool** в прямом проходе усредняет. В обратном проходе он берет градиент и **равномерно распределяет** его по всем пикселям окна с коэффициентом \(1/K^2\).
3. **Отсутствие параметров:** Мы не обновляем никакие веса. Наша задача — корректно передать \(\delta\) на предыдущий слой.

Теперь у нас есть все блоки для CNN: свёртка, активация (ReLU) и пулинг. Готовы собрать их вместе в единую сеть (Flatten + Dense) и написать полный backprop для всей архитектуры?